In [2]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 18.2 MB/s eta 0:00:00


In [3]:
import os
import re
import random
import pandas as pd
import torch

from PIL import Image

from datasets import load_dataset

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

from qwen_vl_utils import process_vision_info

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [4]:
questions_ds = load_dataset(
    "Rowan/vcr",
    "questions",
    split="train"
)

image_ds = load_dataset(
    "Rowan/vcr",
    "image_examples",
    split="train"
)

print("Questions:", len(questions_ds))
print("Images:", len(image_ds))

README.md:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

questions/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 76.3MB            

questions/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

questions/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 76.8MB            

...

Generating train split:   0%|          | 0/80418 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9929 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9557 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/47 [00:00<?, ?it/s]

Questions: 212923
Images: 80418


In [5]:
import os
import subprocess

print("Largest folders in /content:")
!du -h --max-depth=1 /content 2>/dev/null | sort -hr | head -20

Largest folders in /content:
55M	/content/sample_data
55M	/content
148K	/content/.config


In [6]:
print("Largest folders in /root:")
!du -h --max-depth=2 /root 2>/dev/null | sort -hr | head -30

Largest folders in /root:
69G	/root
60G	/root/.cache/huggingface
60G	/root/.cache
8.7G	/root/.julia
7.5G	/root/.julia/artifacts
929M	/root/.julia/compiled
267M	/root/.julia/packages
74M	/root/.cache/pip
60M	/root/.npm/_cacache
60M	/root/.npm
56M	/root/.cache/node-gyp
11M	/root/.julia/registries
9.9M	/root/.julia/scratchspaces
2.5M	/root/.launchpadlib/api.launchpad.net
2.5M	/root/.launchpadlib
148K	/root/.ipython
136K	/root/.ipython/profile_default
116K	/root/.julia/environments
116K	/root/.config
100K	/root/.config/go
88K	/root/.jupyter
76K	/root/.local
68K	/root/.local/share
64K	/root/.npm/_logs
36K	/root/.cache/matplotlib
28K	/root/.julia/logs
16K	/root/.julia/conda
8.0K	/root/.nv
8.0K	/root/.keras
8.0K	/root/.julia/prefs


In [7]:
!rm -rf /root/.cache/huggingface

In [8]:
image_index = {
    img_fn: i
    for i, img_fn in enumerate(image_ds["img_fn"])
}

print("Images indexed:", len(image_index))

Images indexed: 80418


In [9]:
from torch.utils.data import Dataset


class VCRDataset(Dataset):

    def __init__(self, questions, images, image_index):
        self.questions = questions
        self.images = images
        self.image_index = image_index

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):

        q = self.questions[idx]

        image_idx = self.image_index[q["img_fn"]]
        img = self.images[image_idx]

        return {
            "image": img["image"],
            "question": q["question_text"],
            "answers": q["answer_choice_texts"],
            "answer_label": q["answer_label"],
            "rationales": q["rationale_choice_texts"],
            "rationale_label": q["rationale_label"],
            "img_fn": q["img_fn"],
            "img_id": q["img_id"]
        }


vcr_dataset = VCRDataset(
    questions_ds,
    image_ds,
    image_index
)

print("Dataset:", len(vcr_dataset))

Dataset: 212923


In [10]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(
    model_id
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Model loaded.")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded.


In [30]:
def ask_qwen(image, prompt, max_new_tokens=5):

    image = image.convert("RGB")

    # Keep the image small enough for the 14 GB GPU
    image.thumbnail((448, 448))

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(
            inputs.input_ids,
            generated_ids
        )
    ]

    output = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0]

    del inputs
    del generated_ids
    del generated_ids_trimmed

    torch.cuda.empty_cache()

    return output.strip()

In [31]:
def extract_choice(output):

    match = re.search(
        r"\b([0-3])\b",
        output
    )

    if match:
        return int(match.group(1))

    return -1

In [32]:
NUM_SAMPLES = 100

random.seed(42)

sample_indices = random.sample(
    range(len(vcr_dataset)),
    NUM_SAMPLES
)

print("Samples:", len(sample_indices))

Samples: 100


In [33]:
def baseline_prompt(question, answers):

    return f"""
Answer the following question about the image.

Question:
{question}

Answer choices:
0. {answers[0]}
1. {answers[1]}
2. {answers[2]}
3. {answers[3]}

Return only the number of the correct answer.
"""

In [34]:
def improved_prompt(question, answers):

    return f"""
You are solving a visual question.

Look carefully at the image and understand the situation before choosing an answer.

Use:
- the visual information in the image
- the relationship between people and objects
- what is happening in the scene
- common sense

Question:
{question}

Answer choices:
0. {answers[0]}
1. {answers[1]}
2. {answers[2]}
3. {answers[3]}

Choose the single best answer.

Return only:
0, 1, 2, or 3.
"""

In [35]:
def few_shot_prompt(question, answers):

    return f"""
You are solving Visual Commonsense Reasoning questions.

Example 1:

Question:
What is the person likely doing?

Choices:
0. Sleeping
1. Cooking
2. Driving
3. Swimming

Correct answer:
1


Example 2:

Question:
What is most likely true about the object?

Choices:
0. It is being used
1. It is underwater
2. It is flying
3. It is broken

Correct answer:
0


Now solve the new question.

Question:
{question}

Choices:
0. {answers[0]}
1. {answers[1]}
2. {answers[2]}
3. {answers[3]}

Return only the number of the best answer.
"""

In [36]:
def reasoning_prompt(question, answers):

    return f"""
Solve this Visual Commonsense Reasoning question.

First, carefully inspect the image.

Then:
1. Identify the important people and objects.
2. Understand their relationships and actions.
3. Understand what the question is asking.
4. Compare all four answer choices.
5. Eliminate answers that do not fit the image or situation.
6. Select the best answer.

Question:
{question}

Answer choices:
0. {answers[0]}
1. {answers[1]}
2. {answers[2]}
3. {answers[3]}

Reason carefully before answering.

Return ONLY the final answer number:
0, 1, 2, or 3.
"""

In [37]:
def candidate_scoring_prompt(question, answers):

    return f"""
You are solving a Visual Commonsense Reasoning question.

Look carefully at the image.

Evaluate each answer choice against the image and the question.

For each candidate, consider:
- Does it match what is visible?
- Does it correctly describe the people or objects involved?
- Does it make sense given the situation?
- Is there another answer that fits better?

Compare all four candidates and select the strongest one.

Question:
{question}

Candidates:

0. {answers[0]}

1. {answers[1]}

2. {answers[2]}

3. {answers[3]}

After comparing the candidates, return ONLY the number of the best candidate.
"""

In [39]:
import random

random.seed(42)

NUM_SAMPLES = 100

sample_indices = random.sample(
    range(len(vcr_dataset)),
    NUM_SAMPLES
)

print("Using", len(sample_indices), "fixed samples")

Using 100 fixed samples


In [40]:
def run_prompt(prompt_name, prompt_function):

    results = []

    print("\n==============================")
    print("Running:", prompt_name)
    print("==============================")

    for count, idx in enumerate(sample_indices):

        sample = vcr_dataset[idx]

        prompt = prompt_function(
            sample["question"],
            sample["answers"]
        )

        output = ask_qwen(
            sample["image"],
            prompt
        )

        prediction = extract_choice(output)

        results.append({
            "index": idx,
            "true_answer": sample["answer_label"],
            "prediction": prediction,
            "correct": prediction == sample["answer_label"],
            "output": output
        })

        if (count + 1) % 10 == 0:
            print(
                f"{count + 1}/{NUM_SAMPLES} completed"
            )

    results_df = pd.DataFrame(results)

    accuracy = results_df["correct"].mean()

    print("\nAccuracy:", f"{accuracy:.2%}")

    torch.cuda.empty_cache()

    return results_df, accuracy

In [41]:
baseline_results, baseline_accuracy = run_prompt(
    "Baseline",
    baseline_prompt
)


Running: Baseline


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


10/100 completed
20/100 completed
30/100 completed
40/100 completed
50/100 completed
60/100 completed
70/100 completed
80/100 completed
90/100 completed
100/100 completed

Accuracy: 62.00%


In [42]:
improved_results, improved_accuracy = run_prompt(
    "Improved",
    improved_prompt
)


Running: Improved
10/100 completed
20/100 completed
30/100 completed
40/100 completed
50/100 completed
60/100 completed
70/100 completed
80/100 completed
90/100 completed
100/100 completed

Accuracy: 64.00%


In [43]:
few_shot_results, few_shot_accuracy = run_prompt(
    "Few-shot",
    few_shot_prompt
)


Running: Few-shot
10/100 completed
20/100 completed
30/100 completed
40/100 completed
50/100 completed
60/100 completed
70/100 completed
80/100 completed
90/100 completed
100/100 completed

Accuracy: 50.00%


In [44]:
reasoning_results, reasoning_accuracy = run_prompt(
    "Reasoning",
    reasoning_prompt
)


Running: Reasoning
10/100 completed
20/100 completed
30/100 completed
40/100 completed
50/100 completed
60/100 completed
70/100 completed
80/100 completed
90/100 completed
100/100 completed

Accuracy: 63.00%


In [45]:
candidate_results, candidate_accuracy = run_prompt(
    "Candidate Scoring",
    candidate_scoring_prompt
)


Running: Candidate Scoring
10/100 completed
20/100 completed
30/100 completed
40/100 completed
50/100 completed
60/100 completed
70/100 completed
80/100 completed
90/100 completed
100/100 completed

Accuracy: 61.00%


In [46]:
comparison = pd.DataFrame({
    "Prompt": [
        "Baseline",
        "Improved",
        "Few-shot",
        "Reasoning",
        "Candidate Scoring"
    ],
    "Accuracy": [
        baseline_accuracy,
        improved_accuracy,
        few_shot_accuracy,
        reasoning_accuracy,
        candidate_accuracy
    ]
})

comparison["Accuracy"] = comparison["Accuracy"] * 100

comparison = comparison.sort_values(
    "Accuracy",
    ascending=False
)

comparison

,Prompt,Accuracy
1,Improved,64.0
3,Reasoning,63.0
0,Baseline,62.0
4,Candidate Scoring,61.0
2,Few-shot,50.0
